# hive-bot training

Runs the self-play/train iterate loop (see `src/hive_bot/training/train.py`) with checkpoints saved to Google Drive so training can be resumed across separate Colab sessions.

Runtime: Runtime > Change runtime type > GPU is optional but speeds up the network forward passes MCTS relies on -- the engine itself (move generation, apply/undo) is plain Python/CPU regardless.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

## 1. Install hive-bot

Pick ONE of the two cells below depending on where this repo lives.

- **Git remote** (recommended once pushed somewhere): set `REPO_URL` and run the first cell.
- **Local copy on Google Drive**: skip the git cell, mount Drive (next section), and run the second cell pointing at the mounted path instead.

Either way, this only needs the `hive_bot` package itself -- no Django/pydantic dependency, so a bare Colab runtime is enough.

In [ ]:
# Option A: install from a git remote.
REPO_URL = "https://github.com/JODLYO/hive-bot.git"

!git clone {REPO_URL} /content/hive-bot 2>/dev/null || (cd /content/hive-bot && git pull)
!pip install -q /content/hive-bot

In [ ]:
# Option B: install a local copy already sitting on Google Drive (mount
# Drive first -- see the next section -- then point this at the repo root).
# LOCAL_REPO_PATH = "/content/drive/MyDrive/hive-bot"
# !pip install -q {LOCAL_REPO_PATH}

## 2. Mount Google Drive

Checkpoints go here so they survive a Colab session ending -- training resumes from the latest one automatically (see `train()`'s `resume_from`).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT_DIR = Path("/content/drive/MyDrive/hive-bot-checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Configuration

Start modest (small network, few simulations) to confirm everything runs, then scale up -- self-play cost grows roughly linearly with `simulations` and `games_per_run`, and the network forward pass dominates per-simulation cost.

In [ ]:
from hive_bot.engine.constants import BASE_PIECE_TYPES

ENABLED_TYPES = (
    BASE_PIECE_TYPES  # base game only for v1; add expansion pieces once this works
)

RUNS_PER_CELL_EXECUTION = 5  # self-play/train iterations per time you run the training cell
GAMES_PER_ITERATION = 20
SIMULATIONS_PER_MOVE = 200
BATCH_SIZE = 128
BATCHES_PER_ITERATION = 100
LEARNING_RATE = 1e-3

# HiveNet(**NETWORK_KWARGS) -- {} uses the real default size (64 channels,
# 6 residual blocks). Shrink this for a faster (weaker) first run.
NETWORK_KWARGS: dict = {}

## 4. Resume (or start) the model

Finds the highest-numbered checkpoint already in `CHECKPOINT_DIR` and resumes from it (model + optimizer state); starts fresh if there isn't one yet.

In [ ]:
import re


def latest_checkpoint(checkpoint_dir: Path) -> Path | None:
    checkpoints = list(checkpoint_dir.glob("checkpoint_*.pt"))
    if not checkpoints:
        return None
    return max(
        checkpoints, key=lambda p: int(re.search(r"checkpoint_(\d+)\.pt", p.name).group(1))
    )


resume_from = latest_checkpoint(CHECKPOINT_DIR)
print("resuming from:", resume_from if resume_from else "(nothing yet -- starting fresh)")

## 5. Train

Re-run this cell as many times as you like (in this session or a future one) -- each run picks up from `CHECKPOINT_DIR`'s latest checkpoint and does `RUNS_PER_CELL_EXECUTION` more iterations.

In [ ]:
from hive_bot.model.network import HiveNet
from hive_bot.training.train import train

model = train(
    iterations=RUNS_PER_CELL_EXECUTION,
    games_per_iter=GAMES_PER_ITERATION,
    simulations=SIMULATIONS_PER_MOVE,
    batch_size=BATCH_SIZE,
    batches_per_iter=BATCHES_PER_ITERATION,
    lr=LEARNING_RATE,
    enabled_types=ENABLED_TYPES,
    checkpoint_dir=CHECKPOINT_DIR,
    resume_from=resume_from,
    model=HiveNet(**NETWORK_KWARGS),
)

# So re-running this cell (or the one above) continues from here, not from
# what `resume_from` pointed at when the notebook started.
resume_from = latest_checkpoint(CHECKPOINT_DIR)

## 6. Try the current model

`HiveBot.analyze` runs MCTS with the trained network and returns a best move, a win probability, and the full ranked move list -- the "what's the best move here" readout the project is for. `num_simulations` here can be much higher than during self-play (this only runs on demand, not for every ply of every training game).

In [ ]:
from hive_bot.analysis.bot import HiveBot
from hive_bot.engine.state import GameState

bot = HiveBot(model, num_simulations=800)
state = GameState.new_game(ENABLED_TYPES)

analysis = bot.analyze(state)
print(f"win probability for the player to move: {analysis.win_probability:.1%}")
print("top moves:")
for evaluation in analysis.move_evaluations[:5]:
    print(f"  {evaluation.move}  (visited {evaluation.visit_fraction:.1%} of the time)")